# Introduction

Time series analysis has turned out to be significantly more difficult than I originally anticipated. As a technically-oriented trader, I have worked with time series data in various contexts. However, data analysis and forecasting differ quite a bit from the tasks typically involved in trading, at least from my point of view. The time required for each step of time series modeling has also been much greater than I expected.

In this notebook, I applied several models, including LightGBM, Prophet, and LSTM. Unfortunately, none of them delivered particularly strong results. This suggests that there may be issues either in my modeling approach or in how the data was processed. Despite the subpar performance, I have decided to document the problems here as a first attempt. I hope to revisit this project in the future to better understand the limitations of these models and to explore more effective ways of handling temporal relationships in time series data.

# Scenario

All the information you should know is [here](https://www.kaggle.com/competitions/store-sales-time-series-forecasting).

# Data Exploration and Analysis

The original dataset consists of several related tables. My first task was to merge these tables in an appropriate way to prepare a unified dataset for the modeling phase that follows.

In [ ]:
import pandas as pd

In [ ]:
df_train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
df_test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')
df_holidays_events = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
df_oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
df_stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
df_transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')


In [ ]:
df = pd.concat([df_train, df_test], ignore_index=True)


Since this is a time series task, a continuous and complete timeline is essential for certain models. After examining the original data, I found that some dates were missing from the time axis. Specifically, I manually filled in the missing entries for four Christmas days and added the corresponding combinations of store_nbr and family to ensure completeness across the entire time series.

In [ ]:
import pandas as pd

# 原始数据
df["date"] = pd.to_datetime(df["date"])

# 指定要补的圣诞节日期
christmas_dates = pd.to_datetime([
    "2013-12-25", "2014-12-25", "2015-12-25", "2016-12-25"
])

# 所有组合
store_family_combos = df[["store_nbr", "family"]].drop_duplicates()

# 创建4个日期 × 所有组合 的新表（cross merge）
df_christmas = store_family_combos.merge(
    pd.DataFrame({"date": christmas_dates}),
    how="cross"
)

# 添加默认值
df_christmas["sales"] = 0.0
df_christmas["onpromotion"] = 0

# 合并原始数据 + 补上的行（去重）
df_step_time = pd.concat([df, df_christmas], ignore_index=True)
df_step_time = df_step_time.drop_duplicates(subset=["date", "store_nbr", "family"])



After addressing the missing dates, I merged the store information into the main dataset to incorporate store-level attributes for modeling.

In [ ]:
import pandas as pd

# 读取并检查数据
df_stores.columns = df_stores.columns.str.strip()  # 防止列名有空格
df_stores["store_nbr"] = df_stores["store_nbr"].astype(int)

# 确保主表中的 store_nbr 格式一致
df_step_time["store_nbr"] = df_step_time["store_nbr"].astype(int)

# 合并门店信息到主表 df 中
df_step_store = df_step_time.merge(df_stores, on="store_nbr", how="left")


Next, I merged the holiday information into the main dataset. Although the holiday data could be further categorized or refined, I did not pursue that in this version of the analysis. While I do not speak Spanish, it is apparent from the description field that some holidays span multiple days, with the actual holiday date at the center and additional days before and after likely being part of the broader holiday period. I believe this temporal range may have an impact on sales behavior. If time and suitable methods allow, this aspect could be explored more thoroughly in future iterations.

In [ ]:
import pandas as pd

# 假设 df_step_store 和 df_holidays_events 已经加载

# 统一日期格式
df_step_store['date'] = pd.to_datetime(df_step_store['date'])
df_holidays_events['date'] = pd.to_datetime(df_holidays_events['date'])

# 重命名 holidays 表中的 type 列，避免冲突
df_holidays_events = df_holidays_events.rename(columns={'type': 'type_holi'})

# 按 locale 分类
df_local = df_holidays_events[df_holidays_events['locale'] == 'Local'].copy()
df_regional = df_holidays_events[df_holidays_events['locale'] == 'Regional'].copy()
df_national = df_holidays_events[df_holidays_events['locale'] == 'National'].copy()

# 聚合函数，设置 as_index=False 防止索引列冲突
def aggregate_holidays(df, key_cols):
    return (
        df.groupby(key_cols, as_index=False)
          .agg({
              'type_holi': lambda x: '; '.join(x.astype(str).unique()),
              'locale': lambda x: '; '.join(x.astype(str).unique()),
              'locale_name': lambda x: '; '.join(x.astype(str).unique()),
              'description': lambda x: '; '.join(x.astype(str).unique()),
              'transferred': lambda x: '; '.join(x.astype(str).unique())
          })
    )

# 聚合节日数据
agg_local = aggregate_holidays(df_local, ['date', 'locale_name']).rename(columns={'locale_name': 'city'})
agg_regional = aggregate_holidays(df_regional, ['date', 'locale_name']).rename(columns={'locale_name': 'state'})
agg_national = aggregate_holidays(df_national, ['date'])

# 合并函数，避免列冲突
def safe_get(df, col, length):
    if col in df.columns:
        return df[col]
    else:
        return pd.Series([pd.NA] * length)

# 按照 city/state/date 合并
local_merged = pd.merge(
    df_step_store,
    agg_local,
    how='left',
    left_on=['date', 'city'],
    right_on=['date', 'city']
)

regional_merged = pd.merge(
    df_step_store,
    agg_regional,
    how='left',
    left_on=['date', 'state'],
    right_on=['date', 'state']
)

national_merged = pd.merge(
    df_step_store,
    agg_national,
    how='left',
    on='date'
)

# 先复制一份原数据
merged = df_step_store.copy()
n = len(merged)

# 按优先级合并节日信息（Local > Regional > National）
for col in ['type_holi', 'locale', 'locale_name', 'description', 'transferred']:
    local_col = safe_get(local_merged, col, n)
    regional_col = safe_get(regional_merged, col, n)
    national_col = safe_get(national_merged, col, n)

    merged[col] = local_col.combine_first(regional_col).combine_first(national_col)

# 删除 description 列
merged.drop(columns=['description'], inplace=True)

df_step_holidays = merged



Following that, I incorporated oil price data into the dataset. For weekends, where oil prices are not reported, I followed the convention used in the competition and carried forward Friday’s price to fill in the missing values.

In [ ]:
import pandas as pd

def fill_oil_prices(df_oil):
    # 确保 date 是列且为 datetime
    if 'date' not in df_oil.columns:
        # date 可能在索引，先重置索引
        df_oil = df_oil.reset_index()
    df_oil['date'] = pd.to_datetime(df_oil['date'])

    # 排序并设置索引方便补全
    df_oil = df_oil.sort_values('date').set_index('date')

    # 补全日期索引
    full_idx = pd.date_range(start=df_oil.index.min(), end=df_oil.index.max(), freq='D')

    # 重新索引并向前填充缺失值
    df_oil_full = df_oil.reindex(full_idx).ffill()

    # 重置索引，把日期从索引变成列
    df_oil_full = df_oil_full.reset_index().rename(columns={'index': 'date'})

    return df_oil_full

# 使用示范
df_oil_full = fill_oil_prices(df_oil)
df_oil_full.loc[df_oil_full['date'] == '2013-01-01', 'dcoilwtico'] = 93.14


In [ ]:
# 确保日期格式正确
df_step_holidays['date'] = pd.to_datetime(df_step_holidays['date'])
df_oil_full['date'] = pd.to_datetime(df_oil_full['date'])

# 左连接合并
df_step_oil = pd.merge(
    df_step_holidays,
    df_oil_full,
    how='left',
    on='date'
)

# df_step_oil 就是最终合并结果


Finally, I merged the transaction count data into the main dataset. With this step completed, the raw data preprocessing was finished.

In [ ]:
# 确保日期列是 datetime 类型
df_step_oil['date'] = pd.to_datetime(df_step_oil['date'])
df_transactions['date'] = pd.to_datetime(df_transactions['date'])

# 左连接合并
df_step_trans = pd.merge(
    df_step_oil,
    df_transactions,
    how='left',
    on=['date', 'store_nbr']
)


In [ ]:
df_step_trans['transactions'] = df_step_trans['transactions'].fillna(0).astype(int)


In [ ]:
# 1. 创建是否为节日的标志列（非缺失为1，缺失为0）
df_step_trans['has_holiday'] = df_step_trans['type_holi'].notna().astype(int)

# 2. 填充缺失值
df_step_trans['type_holi'] = df_step_trans['type_holi'].fillna('Other')
df_step_trans['locale'] = df_step_trans['locale'].fillna('Other')
df_step_trans['transferred'] = df_step_trans['transferred'].fillna('Other')

In [ ]:
df_step_trans = df_step_trans.drop(columns=['locale_name'])


In [ ]:
df_full = df_step_trans.copy()

# Model Building

First, I experimented with the traditional tree-based model LightGBM. In theory, tree models are well-suited for time series tasks, but in this case, the actual performance was very poor. This may be due to issues with my data preprocessing or other factors. 

In [ ]:
#import pandas as pd
#import numpy as np
#from sklearn.model_selection import KFold
#from sklearn.preprocessing import LabelEncoder
#import lightgbm as lgb
#from tqdm import tqdm
#import gc

# ✅ 拷贝原始数据，保留 df_full 不变
#df_light = df_full.copy()

# ✅ 日期拆分特征
#df_light["date"] = pd.to_datetime(df_light["date"])
#df_light["year"] = df_light["date"].dt.year
#df_light["month"] = df_light["date"].dt.month
#df_light["day"] = df_light["date"].dt.day
#df_light["dayofweek"] = df_light["date"].dt.dayofweek

# ✅ Label Encoding 处理所有类别变量
#cat_cols = ["store_nbr", "family", "city", "state", "type", "cluster", "type_holi", "locale"]
#for col in cat_cols:
#    if col in df_light.columns:
#        df_light[col] = LabelEncoder().fit_transform(df_light[col].astype(str))

# ✅ bool 类型处理（如 transferred）
#if "transferred" in df_light.columns:
#    df_light["transferred"] = df_light["transferred"].replace({"False": False, "True": True})
#    df_light["transferred"] = df_light["transferred"].astype(bool).astype(int)

# ✅ 分出 train/test（注意保留 test 中的 id）
#df_test = df_light[df_light["sales"].isna()].reset_index(drop=True)
#df_train = df_light[df_light["sales"].notna()].reset_index(drop=True)

# ✅ 删除 train 中的 id（test 中保留）
#if "id" in df_train.columns:
#    df_train = df_train.drop(columns=["id"])

# ✅ 特征和目标列
#features = [col for col in df_train.columns if col not in ["sales", "date"]]
#target = "sales"

# ✅ 7折交叉验证
#kf = KFold(n_splits=7, shuffle=True, random_state=42)
#preds = np.zeros(len(df_test))

#print("Training LightGBM with 7-fold CV:")
#for fold, (train_idx, val_idx) in enumerate(kf.split(df_train), 1):
#    train_data = lgb.Dataset(df_train.loc[train_idx, features], label=df_train.loc[train_idx, target],
#                             categorical_feature=cat_cols)
#    val_data = lgb.Dataset(df_train.loc[val_idx, features], label=df_train.loc[val_idx, target],
#                           categorical_feature=cat_cols)

#    params = {
#        "objective": "regression",
#        "metric": "rmse",
#        "verbosity": -1,
#        "boosting_type": "gbdt",
#        "seed": 42,
#        "learning_rate": 0.05,
#        "num_leaves": 31,
#        "feature_fraction": 0.8,
#        "bagging_fraction": 0.8,
#        "bagging_freq": 5,
#    }

#    model = lgb.train(
#        params,
#        train_data,
#        valid_sets=[val_data],
#        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
#    )

#    preds += model.predict(df_test[features], num_iteration=model.best_iteration) / kf.n_splits
#    print(f"✅ Fold {fold}/7 done. Progress: {fold/7:.2%}")

# ✅ 创建提交文件（保留 id）
#submission = df_test[["id"]].copy()
#submission["sales"] = preds
#submission["sales"] = submission["sales"].clip(lower=0)
#submission["id"] = submission["id"].astype(int)

#submission.to_csv("submission.csv", index=False)

# ✅ 清理变量，释放内存
#del df_light, df_train, df_test, train_data, val_data, model
#gc.collect()


Next, I attempted a more advanced model, LSTM, but the results were still very poor. Clearly, I must have made some critical mistakes in certain steps.


In [ ]:
#import pandas as pd
#import numpy as np
#import torch
#import torch.nn as nn
#from torch.utils.data import Dataset, DataLoader
#from sklearn.preprocessing import LabelEncoder
#from sklearn.model_selection import train_test_split
#import gc

# === 参数 ===
#SEQ_LEN = 7
#BATCH_SIZE = 64
#EPOCHS = 20
#LR = 0.001
#DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === 数据准备 ===
#data_full = df_full.copy()
#data_full["date"] = pd.to_datetime(data_full["date"])

# Label Encoding
#cat_cols = ["store_nbr", "family", "city", "state", "type", "cluster", "type_holi", "locale"]
#for col in cat_cols:
#    if col in data_full.columns:
#        data_full[col] = LabelEncoder().fit_transform(data_full[col].astype(str))

# Bool转int
#if "transferred" in data_full.columns:
#    data_full["transferred"] = data_full["transferred"].replace({"False": False, "True": True}).astype(bool).astype(int)

# 特征列
#feature_cols = ["sales", "onpromotion"] + cat_cols

# 分训练和测试集
#data_train = data_full[data_full["sales"].notna()].copy()
#data_test = data_full[data_full["sales"].isna()].copy()

# 排序
#data_train = data_train.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

# === Dataset构造 ===
#def create_sequences(data, seq_len=SEQ_LEN):
#    xs, ys = [], []
#    for i in range(len(data) - seq_len):
#        x = data.iloc[i:i+seq_len]
#        y = data.iloc[i+seq_len]["sales"]
#        xs.append(x)
#        ys.append(y)
#    return xs, ys

#class SalesDataset(Dataset):
#    def __init__(self, df, seq_len):
#        self.seq_len = seq_len
#        self.features = []
#        self.targets = []
#        grouped = df.groupby(["store_nbr", "family"])
#        for _, group in grouped:
#            group = group.reset_index(drop=True)
#            xs, ys = create_sequences(group[feature_cols], seq_len)
#            self.features.extend(xs)
#            self.targets.extend(ys)
#    def __len__(self):
#        return len(self.targets)
#    def __getitem__(self, idx):
#        x = torch.tensor(self.features[idx].values, dtype=torch.float32)
#        y = torch.tensor(self.targets[idx], dtype=torch.float32)
#        return x, y

# 训练/验证划分
#train_df, val_df = train_test_split(data_train, test_size=0.15, shuffle=False, random_state=42)
#train_dataset = SalesDataset(train_df, SEQ_LEN)
#val_dataset = SalesDataset(val_df, SEQ_LEN)
#train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
#val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# === LSTM模型定义 ===
#class SalesLSTM(nn.Module):
#    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
#        super().__init__()
#        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
#        self.fc = nn.Linear(hidden_dim, 1)
#    def forward(self, x):
#        out, _ = self.lstm(x)
#        out = out[:, -1, :]
#        out = self.fc(out)
#        return out.squeeze()

#input_dim = len(feature_cols)
#model = SalesLSTM(input_dim).to(DEVICE)
#criterion = nn.MSELoss()
#optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# === 训练函数 ===
#def train_epoch(model, dataloader, optimizer, criterion):
#    model.train()
#    total_loss = 0
#    for x, y in dataloader:
#        x, y = x.to(DEVICE), y.to(DEVICE)
#        optimizer.zero_grad()
#        pred = model(x)
#        loss = criterion(pred, y)
#        loss.backward()
#        optimizer.step()
#        total_loss += loss.item() * x.size(0)
#    return total_loss / len(dataloader.dataset)

#def eval_epoch(model, dataloader, criterion):
#    model.eval()
#    total_loss = 0
#    with torch.no_grad():
#        for x, y in dataloader:
#            x, y = x.to(DEVICE), y.to(DEVICE)
#            pred = model(x)
#            loss = criterion(pred, y)
#            total_loss += loss.item() * x.size(0)
#    return total_loss / len(dataloader.dataset)

# === 模型训练 ===
#best_val_loss = np.inf
#for epoch in range(1, EPOCHS + 1):
#    train_loss = train_epoch(model, train_loader, optimizer, criterion)
#    val_loss = eval_epoch(model, val_loader, criterion)
#    print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
#    if val_loss < best_val_loss:
#        best_val_loss = val_loss
#        torch.save(model.state_dict(), "best_lstm.pth")

# === 滚动预测测试集所有天数 ===
#model.load_state_dict(torch.load("best_lstm.pth"))
#model.eval()

#submission_rows = []

#for (store, fam), test_group in data_test.groupby(["store_nbr", "family"]):
#    test_group = test_group.sort_values("date")
#    train_group = data_train[(data_train["store_nbr"]==store) & (data_train["family"]==fam)].sort_values("date")

#    if len(train_group) < SEQ_LEN:
#        continue

#    data_seq = train_group.iloc[-SEQ_LEN:].copy()

#    for _, row in test_group.iterrows():
#        input_seq = data_seq[feature_cols].copy()
#        x = torch.tensor(input_seq.values, dtype=torch.float32).unsqueeze(0).to(DEVICE)

#        with torch.no_grad():
#            y_pred = model(x).item()

#        # 构造新的一天数据
#        new_row = row[feature_cols].copy()
#        new_row["sales"] = y_pred
#        data_seq = pd.concat([data_seq, pd.DataFrame([new_row])], ignore_index=True).iloc[1:]

#        submission_rows.append({"id": int(row["id"]), "sales": max(y_pred, 0)})

# === 保存提交结果 ===
#submission = pd.DataFrame(submission_rows)
#submission.to_csv("submission.csv", index=False)
#print("✅ 所有测试集预测完成：submission_lstm.csv")

# === 清理内存 ===
#del model, train_loader, val_loader, train_dataset, val_dataset
#gc.collect()


Finally, I tried the Prophet model, which is specifically designed for time series analysis. The results were much better this time, although I did not perform extensive parameter tuning.

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import gc

df_prophet = df_full.copy()
df_prophet["date"] = pd.to_datetime(df_prophet["date"])

# 处理节假日，保留字符串，不编码
if "type_holi" in df_full.columns:
    holidays = df_full[["date", "type_holi"]].dropna().drop_duplicates().copy()
    holidays = holidays.rename(columns={"date": "ds", "type_holi": "holiday"})
    holidays["holiday"] = holidays["holiday"].astype(str)
else:
    holidays = None

# Label Encoding（type_holi不编码）
cat_cols = ["store_nbr", "family", "city", "state", "type", "cluster", "locale"]
for col in cat_cols:
    if col in df_prophet.columns:
        df_prophet[col] = LabelEncoder().fit_transform(df_prophet[col].astype(str))

# 布尔转数值
if "transferred" in df_prophet.columns:
    df_prophet["transferred"] = df_prophet["transferred"].replace({"False": False, "True": True})
    df_prophet["transferred"] = df_prophet["transferred"].astype(bool).astype(int)

# 日期拆分
df_prophet["year"] = df_prophet["date"].dt.year
df_prophet["month"] = df_prophet["date"].dt.month
df_prophet["day"] = df_prophet["date"].dt.day
df_prophet["dayofweek"] = df_prophet["date"].dt.dayofweek

# 划分训练测试
df_test = df_prophet[df_prophet["sales"].isna()].reset_index(drop=True)
df_train = df_prophet[df_prophet["sales"].notna()].reset_index(drop=True)

group_cols = ["store_nbr", "family"]

regressors = [
    "city", "state", "type", "cluster", "locale",
    "onpromotion", "dcoilwtico", "transferred",
    "year", "month", "day", "dayofweek"
]
regressors = [c for c in regressors if c in df_prophet.columns]

print(f"使用的回归变量: {regressors}")

for keys, group_df in tqdm(df_train.groupby(group_cols), total=df_train[group_cols].drop_duplicates().shape[0]):
    test_mask = (df_test[group_cols[0]] == keys[0]) & (df_test[group_cols[1]] == keys[1])
    df_test_group = df_test[test_mask].copy()
    if df_test_group.empty:
        continue

    df_train_group = group_df[["date", "sales"] + regressors].copy()
    df_train_group = df_train_group.rename(columns={"date": "ds", "sales": "y"})

    model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False, holidays=holidays)
    for reg in regressors:
        model.add_regressor(reg)

    try:
        model.fit(df_train_group)
    except Exception as e:
        print(f"训练错误 group={keys}: {e}")
        continue

    df_test_group = df_test_group[["date"] + regressors].copy()
    df_test_group = df_test_group.rename(columns={"date": "ds"})

    try:
        forecast = model.predict(df_test_group)
        df_test.loc[test_mask, "sales"] = forecast["yhat"].values
    except Exception as e:
        print(f"预测错误 group={keys}: {e}")
        continue

submission = df_test[["id", "sales"]].copy()
submission["sales"] = submission["sales"].clip(lower=0)
submission["id"] = submission["id"].astype(int)

submission.to_csv("submission.csv", index=False)
print("已保存 submission_prophet_full.csv")

del df_prophet, df_train, df_test
gc.collect()


At the very end, I attempted to enhance the Prophet model by adding some features to improve the final prediction, but the results still did not surpass the original performance.

In [ ]:
#pip install neuralprophet


In [ ]:
#import pandas as pd
#import numpy as np
#from neuralprophet import NeuralProphet
#from sklearn.preprocessing import LabelEncoder
#from tqdm import tqdm
#import multiprocessing as mp
#import gc

#df_neu = df_full.copy()
#df_neu["date"] = pd.to_datetime(df_neu["date"])

# Label encoding
#cat_cols = ["store_nbr", "family", "city", "state", "type", "cluster", "locale"]
#for col in cat_cols:
#    if col in df_neu.columns:
#        df_neu[col] = LabelEncoder().fit_transform(df_neu[col].astype(str))

# 转换布尔
#if "transferred" in df_neu.columns:
#    df_neu["transferred"] = df_neu["transferred"].replace({"False": False, "True": True})
#    df_neu["transferred"] = df_neu["transferred"].astype(bool).astype(int)

# 日期特征
#df_neu["year"] = df_neu["date"].dt.year
#df_neu["month"] = df_neu["date"].dt.month
#df_neu["day"] = df_neu["date"].dt.day
#df_neu["dayofweek"] = df_neu["date"].dt.dayofweek

# 分割数据
#df_test = df_neu[df_neu["sales"].isna()].reset_index(drop=True)
#df_train = df_neu[df_neu["sales"].notna()].reset_index(drop=True)

#group_cols = ["store_nbr", "family"]

#regressors = [
#    "city", "state", "type", "cluster", "locale",
#    "onpromotion", "dcoilwtico", "transferred",
#    "year", "month", "day", "dayofweek"
#]
#regressors = [c for c in regressors if c in df_neu.columns]

#print(f"使用的回归变量: {regressors}")


#def process_group(keys):
#    store_nbr, family = keys
#    test_mask = (df_test["store_nbr"] == store_nbr) & (df_test["family"] == family)
#    df_test_group = df_test[test_mask].copy()
#    if df_test_group.empty:
#        return None

#    group_df = df_train[(df_train["store_nbr"] == store_nbr) & (df_train["family"] == family)]
#    df_train_group = group_df[["date", "sales"] + regressors].copy()
#    df_train_group = df_train_group.rename(columns={"date": "ds", "sales": "y"})

#    model = NeuralProphet(
#        yearly_seasonality=True,
#        weekly_seasonality=True,
#        daily_seasonality=False,
#        epochs=100,
#        learning_rate=1.0,
#    )

#    for reg in regressors:
#        model.add_lagged_regressor(reg)

#    try:
#        model.fit(df_train_group, freq="D", progress="off")
#    except Exception as e:
#        print(f"训练错误 group={keys}: {e}")
#        return None

#    df_test_group = df_test_group[["id", "date"] + regressors].copy()
#    df_test_group = df_test_group.rename(columns={"date": "ds"})

#    try:
#        forecast = model.predict(df_test_group)
#        df_test_group["sales"] = forecast["yhat1"].values
#        df_test_group = df_test_group[["id", "sales"]]
#        return df_test_group
#    except Exception as e:
#        print(f"预测错误 group={keys}: {e}")
#        return None


# 多进程运行
#unique_groups = df_train[group_cols].drop_duplicates().values.tolist()
#with mp.Pool(processes=mp.cpu_count()) as pool:
#    results = list(tqdm(pool.imap(process_group, unique_groups), total=len(unique_groups)))

# 合并所有结果
#submission = pd.concat([r for r in results if r is not None], axis=0)
#submission["sales"] = submission["sales"].clip(lower=0)
#submission["id"] = submission["id"].astype(int)
#submission = submission.sort_values("id")

#submission.to_csv("submission.csv", index=False)
#print("已保存 submission.csv")

#del df_neu, df_train, df_test
#gc.collect()


# Conclusion

As a trader by background, I have my own perspective on time series analysis, which likely differs significantly from the mainstream approaches and the design of popular time series models. I need to absorb the correct mainstream methodologies while retaining some of the ideas I personally find valuable. Naturally, this is something I would focus on after exploring most areas of data science and data analysis, with a dedicated interest in time series.

In the past, while working as a trader and preparing for actuarial exams, I briefly studied some books on time series. However, many people say that there is currently no definitive handbook written by a true expert that can serve as a classic guide in this field, though I am not sure how true that is.

In any case, time series analysis has proven to be much more challenging than I initially imagined. It is no longer a matter of simply switching models and easily improving performance. The path ahead requires deeper understanding and more refined techniques.